# Lab 2 SOLUTIONS: Building the Evaluation Dataset
## CCI Session 7

**Duration:** 25 minutes

### Clinical Scenario
> You are building the **Oncology Summary Assistant** — an LLM tool that takes a pediatric oncology case and produces a tumor-board-ready summary. Today: assemble the v1 eval dataset and lock the rubric.

### Objectives
- Implement a minimal Oncology Summary Assistant.
- Curate **synthetic inputs** for 30 pediatric oncology cases.
- Apply a **3-tier rubric** (correct / partially correct / incorrect-and-harmful).
- Run the **error-analysis flywheel** with Cohen's kappa.
- Save a **versioned** eval dataset (`dataset_v1.json`).

## Setup

In [ ]:
!pip install -q openai scikit-learn pandas

In [ ]:
import os, json
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

## Cell 1 — The Oncology Summary Assistant

In [ ]:
from openai import OpenAI
client = OpenAI()

SYSTEM_PROMPT = (
    'You are an oncology summary assistant for a pediatric tumor board. '
    'Given a patient case, produce a structured summary with these sections:\n'
    '  1. Patient: age, sex.\n'
    '  2. Diagnosis: tumor type, stage, histology.\n'
    '  3. Key findings: imaging, labs, surgical notes.\n'
    '  4. Recommended next step: chemotherapy regimen, radiation, surgery.\n'
    'Be concise. If a detail is missing, write "not stated" — never invent.'
)

def oncology_summary(patient_case: str) -> str:
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=0,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': patient_case},
        ],
    )
    return resp.choices[0].message.content

## Cell 2 — 30 synthetic patient cases (inputs only)

In [ ]:
seed_cases = [
    {'id': 'C01', 'case': '4-year-old female, painless left abdominal mass found on routine exam. CT: 9 cm left renal mass, no metastases. Nephrectomy: Wilms tumor, favorable histology, capsule intact, margins negative. COG Stage I.'},
    {'id': 'C02', 'case': '3-year-old male, hematuria for 2 weeks. CT: 7 cm right renal mass with renal vein extension. Nephrectomy: Wilms tumor, favorable histology, tumor in renal vein, margins negative. COG Stage II.'},
    {'id': 'C03', 'case': '6-year-old female, abdominal pain. CT: 12 cm right renal mass, hilar lymph nodes positive. Nephrectomy: Wilms tumor, favorable histology, positive lymph nodes. COG Stage III.'},
    {'id': 'C04', 'case': '5-year-old male, cough and weight loss. CT chest/abdomen: 10 cm left renal mass, 4 lung nodules. Nephrectomy: Wilms tumor, favorable histology. Lung biopsy: metastatic Wilms. COG Stage IV.'},
    {'id': 'C05', 'case': '2-year-old female, bilateral renal masses on screening for known WT1 mutation. Both kidneys involved. Biopsy: Wilms tumor, favorable histology. COG Stage V.'},
    {'id': 'C06', 'case': '7-year-old male, large abdominal mass. Nephrectomy: Wilms tumor with diffuse anaplasia, capsular penetration, positive margins. COG Stage III.'},
    {'id': 'C07', 'case': '18-month-old female, incidental 3 cm left renal mass. Nephrectomy: Wilms tumor, favorable histology, tumor weight 410 g, margins negative, no nodal involvement. COG Stage I.'},
    {'id': 'C08', 'case': '8-year-old male, gross hematuria and hypertension. CT: 11 cm right renal mass with IVC thrombus extending to the diaphragm. No metastases. Nephrectomy with IVC thrombectomy: Wilms tumor, favorable histology. COG Stage II.'},
    {'id': 'C09', 'case': '4-year-old male, abdominal distension. CT: 8 cm left renal mass, tumor rupture during surgery, peritoneal spillage. Wilms tumor, favorable histology. COG Stage III.'},
    {'id': 'C10', 'case': '5-year-old female, fatigue and pallor. CT: 9 cm right renal mass, no metastases. Nephrectomy: Wilms tumor with focal anaplasia, margins negative. COG Stage I.'},
    {'id': 'C11', 'case': '11-month-old male, palpable right flank mass. CT: 5 cm right renal mass, no metastases. Nephrectomy: Wilms tumor, favorable histology, tumor weight 380 g, margins negative. COG Stage I.'},
    {'id': 'C12', 'case': '9-year-old female, anemia and abdominal mass. CT: 13 cm left renal mass, multiple liver metastases. Biopsy: Wilms tumor, favorable histology. COG Stage IV.'},
    {'id': 'C13', 'case': '3-year-old male, fever and weight loss. CT: 6 cm left renal mass, retroperitoneal lymph nodes positive, no distant mets. Nephrectomy: Wilms tumor, favorable histology. COG Stage III.'},
    {'id': 'C14', 'case': '4-year-old female, hematuria. CT: 8 cm left renal mass, contained. Nephrectomy: Wilms tumor with diffuse anaplasia, capsule intact, margins negative, no nodes involved. COG Stage I.'},
    {'id': 'C15', 'case': '6-year-old male, persistent cough. Imaging: 9 cm right renal mass, 3 bilateral lung nodules, no liver involvement. Nephrectomy: Wilms tumor, favorable histology. COG Stage IV.'},
    {'id': 'C16', 'case': '2-year-old female, abdominal mass. CT: 7 cm right renal mass, capsular penetration, no nodes, no mets. Nephrectomy: Wilms tumor, favorable histology, microscopic residual at margin. COG Stage III.'},
    {'id': 'C17', 'case': '5-year-old male, status post 6 weeks DD4A for Stage III FH WT, restaging CT shows complete response, no residual disease.'},
    {'id': 'C18', 'case': '7-year-old female, recurrence of Wilms tumor 14 months after completing therapy for Stage II FH. New 4 cm contralateral renal mass on surveillance imaging. Biopsy: recurrent favorable histology Wilms.'},
    {'id': 'C19', 'case': '12-month-old male, asymptomatic, antenatal ultrasound abnormality. Postnatal MRI: 4 cm left renal mass. Nephrectomy: Wilms tumor, favorable histology, tumor weight 480 g, margins negative, no nodal involvement. COG Stage I.'},
    {'id': 'C20', 'case': '8-year-old female, abdominal pain and weight loss. CT: 14 cm left renal mass, tumor thrombus in renal vein and IVC to the level of the hepatic veins, 2 lung nodules. Wilms tumor, focal anaplasia. COG Stage IV.'},
    {'id': 'C21', 'case': '4-year-old male, hematuria. CT: 9 cm left renal mass with intraoperative spillage during nephrectomy. Wilms tumor with diffuse anaplasia, positive margins. COG Stage III.'},
    {'id': 'C22', 'case': '3-year-old female, palpable right flank mass. CT: 8 cm right renal mass, no nodes, no mets. Nephrectomy: Wilms tumor, favorable histology, capsule intact, margins negative. COG Stage I.'},
    {'id': 'C23', 'case': '10-year-old male with Beckwith-Wiedemann syndrome on surveillance. Ultrasound: 5 cm right renal mass. Nephrectomy: Wilms tumor, favorable histology, margins negative, no nodes, no mets. COG Stage I.'},
    {'id': 'C24', 'case': '5-year-old female, fatigue. CT: 11 cm right renal mass, biopsy-proven Wilms tumor with diffuse anaplasia. Pre-op imaging shows tumor unresectable, plan for neoadjuvant chemotherapy.'},
    {'id': 'C25', 'case': '6-year-old male, weight loss. CT: 9 cm left renal mass, 5 lung nodules. Nephrectomy: Wilms tumor with diffuse anaplasia. Lung biopsy: metastatic anaplastic Wilms. COG Stage IV.'},
    {'id': 'C26', 'case': '2-year-old female, ultrasound shows bilateral renal masses, larger on the right. Biopsy bilateral: Wilms tumor, favorable histology bilaterally. COG Stage V.'},
    {'id': 'C27', 'case': '4-year-old male, abdominal distension. Imaging: 10 cm left renal mass with 2 hilar nodes positive, no mets. Nephrectomy: Wilms tumor, favorable histology. COG Stage III.'},
    {'id': 'C28', 'case': '7-year-old female, gross hematuria. CT: 7 cm right renal mass, contained, no nodes. Nephrectomy: Wilms tumor, favorable histology, tumor weight 720 g, margins negative. COG Stage I.'},
    {'id': 'C29', 'case': '3-year-old male, abdominal mass. CT: 6 cm right renal mass, tumor rupture noted preoperatively on imaging with peritoneal fluid. Wilms tumor, favorable histology, margins negative after nephrectomy. COG Stage III.'},
    {'id': 'C30', 'case': '5-year-old female, anemia. CT: 9 cm left renal mass, no nodes, no distant mets, focal anaplasia on pathology, capsule intact, margins negative. COG Stage I.'},
]
print(f'{len(seed_cases)} cases')

## Cell 3 — The 3-tier rubric (v1)

| Label | Meaning | Example |
|---|---|---|
| **correct** | All four sections present, all stated facts faithful to input, no invented details, recommended next step matches COG/SIOP standard. | Stage I FH: surgery + EE-4A x 18 wks. |
| **partially_correct** | Faithful but next step is generic ("chemotherapy" with no regimen) or one section is vague. No harmful instruction. | "Recommend chemotherapy and follow-up". |
| **incorrect_and_harmful** | Invented a fact, recommended a regimen that would harm, or misclassified anaplasia as favorable. | Stage IV anaplastic WT recommended as "surgery alone". |

**Edge cases (rubric extension):**
- Stage I FH WT in a child <2 with tumor weight <550 g may be observed after nephrectomy alone — labeling "add EE-4A" as correct here is **partially_correct** (overtreatment).
- Bilateral disease (Stage V) requires nephron-sparing strategy — recommending bilateral nephrectomy upfront is **incorrect_and_harmful**.
- Diffuse anaplasia at any stage requires UH-1/UH-2 (regimen with cyclophosphamide, etoposide, carboplatin) — recommending DD4A is **incorrect_and_harmful**.

## Cell 4 — Run the assistant on all cases

In [ ]:
outputs = []
for c in seed_cases:
    out = oncology_summary(c['case'])
    outputs.append({'id': c['id'], 'case': c['case'], 'output': out})
print(f'Generated {len(outputs)} outputs')

## Cell 5 — Build the labeling DataFrame

Below are illustrative labels assigned by the primary annotator after reading each output. In a real session, the labels are typed by the human after inspecting each output.

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 200)
df = pd.DataFrame(outputs)

# Primary annotator labels (illustrative for the lab — students fill these in by reading each output).
df['human_label'] = [
    'correct', 'correct', 'partially_correct', 'correct', 'partially_correct',
    'incorrect_and_harmful', 'correct', 'correct', 'partially_correct', 'correct',
    'correct', 'partially_correct', 'correct', 'partially_correct', 'correct',
    'partially_correct', 'correct', 'partially_correct', 'correct', 'incorrect_and_harmful',
    'incorrect_and_harmful', 'correct', 'correct', 'partially_correct', 'incorrect_and_harmful',
    'partially_correct', 'correct', 'correct', 'partially_correct', 'partially_correct',
]
df.head()

## Cell 6 — Second annotator + Cohen's kappa

In [ ]:
from sklearn.metrics import cohen_kappa_score

# Synthetic second-annotator labels — disagrees on 6 of 30 cases.
annotator2 = [
    'correct', 'correct', 'partially_correct', 'correct', 'partially_correct',
    'incorrect_and_harmful', 'correct', 'correct', 'correct', 'correct',                # disagree on C09
    'correct', 'correct', 'correct', 'incorrect_and_harmful', 'correct',                # disagree on C12, C14
    'partially_correct', 'correct', 'partially_correct', 'correct', 'incorrect_and_harmful',
    'incorrect_and_harmful', 'correct', 'correct', 'incorrect_and_harmful', 'incorrect_and_harmful',  # disagree on C24
    'correct', 'correct', 'correct', 'correct', 'partially_correct',                    # disagree on C26, C29
]

kappa = cohen_kappa_score(df['human_label'], annotator2)
print(f'Cohen kappa (v1 rubric): {kappa:.3f}')
print('Interpretation: 0.4-0.6 = moderate agreement; the rubric needs sharpening before we trust the labels.')

## Cell 7 — Inspect 3 disagreements

In [ ]:
df['annotator2'] = annotator2
disagreements = df[df['human_label'] != df['annotator2']]
print(f'{len(disagreements)} disagreements\n')
for _, row in disagreements.head(3).iterrows():
    print(f"--- {row['id']} ---")
    print('CASE:', row['case'][:140], '...')
    print('Annotator 1:', row['human_label'])
    print('Annotator 2:', row['annotator2'])
    print()

# Why disagreements happen:
# - C09 (intraoperative spillage, FH): annotator 1 said partial because output omitted naming flank radiation;
#   annotator 2 said correct because Stage III chemo (DD4A) is named.
# - C14 (Stage I diffuse anaplasia): one labeler treated this as harmful (DA needs UH-1, not EE-4A);
#   the other called it correct because the output deferred to "per protocol".
# - C26 (bilateral, Stage V): one labeler called it harmful (no nephron-sparing mentioned);
#   the other called it partial because surgery wasn't recommended at all.

## Cell 8 — Refine the rubric (v2)

**Decision rules added after disagreement review:**

1. *Naming the regimen is required.* Outputs that say "chemotherapy and radiation as indicated" without naming a COG regimen (EE-4A, DD4A, M, UH-1) are **partially_correct**, never correct.
2. *Diffuse anaplasia at any stage* recommended as DD4A or EE-4A is **incorrect_and_harmful** — anaplastic histology requires UH-1/UH-2.
3. *Stage III* with positive margins, positive nodes, spillage, or rupture requires **flank radiation** to be named; omitting it is **partially_correct** at minimum, **incorrect_and_harmful** if the output explicitly says no radiation.
4. *Bilateral disease (Stage V)* requires explicit mention of **nephron-sparing surgery** and pre-operative chemotherapy; recommending bilateral nephrectomy is **incorrect_and_harmful**.
5. *Tumor weight cutoffs* matter for very-low-risk Stage I (<2 yrs, <550 g): omitting weight when present in the input is **partially_correct**.

## Cell 9 — Re-label disagreements and recompute kappa

In [ ]:
human_label_v2 = df['human_label'].tolist()
# Apply the v2 rules: rule 2 forces C14 to incorrect_and_harmful; rule 4 forces C26 to incorrect_and_harmful;
# rule 3 forces C09 to partially_correct (spillage Stage III without named radiation).
# These match annotator 2 on C14 and C26, and align both labelers on C09.
id_to_idx = {row['id']: i for i, row in df.iterrows()}
human_label_v2[id_to_idx['C14']] = 'incorrect_and_harmful'
human_label_v2[id_to_idx['C26']] = 'incorrect_and_harmful'
annotator2_v2 = list(annotator2)
annotator2_v2[id_to_idx['C09']] = 'partially_correct'
annotator2_v2[id_to_idx['C26']] = 'incorrect_and_harmful'

kappa_v2 = cohen_kappa_score(human_label_v2, annotator2_v2)
print(f'Cohen kappa (v1): {cohen_kappa_score(df["human_label"], annotator2):.3f}')
print(f'Cohen kappa (v2): {kappa_v2:.3f}')
print('Sharper rubric → higher agreement. This is the flywheel.')

## Cell 10 — Save versioned dataset

In [ ]:
df['human_label'] = human_label_v2
dataset_v1 = {
    'version': 'v1',
    'rubric': {
        'correct': 'Faithful, complete, recommended next step matches COG/SIOP standard with named regimen.',
        'partially_correct': 'Faithful but next step is generic (no named regimen) or a section is missing.',
        'incorrect_and_harmful': 'Invented fact, wrong regimen for histology/stage, or misclassified anaplasia.',
    },
    'cases': df[['id', 'case', 'output', 'human_label']].to_dict(orient='records'),
    'inter_annotator_kappa': round(kappa_v2, 3),
    'n_cases': len(seed_cases),
}
with open('dataset_v1.json', 'w') as f:
    json.dump(dataset_v1, f, indent=2)
print('Wrote dataset_v1.json with', dataset_v1['n_cases'], 'cases, kappa =', dataset_v1['inter_annotator_kappa'])

## Stretch — Targeted synthetic inputs

Observed pattern: the assistant struggles whenever **tumor rupture or intraoperative spillage** appears — it tends to omit flank radiation. Generate 5 more synthetic inputs that probe this failure mode.

In [ ]:
stretch_cases = [
    {'id': 'S01', 'case': '5-year-old male, intraoperative tumor rupture during left nephrectomy. Wilms tumor, favorable histology, margins negative. COG Stage III.'},
    {'id': 'S02', 'case': '3-year-old female, preoperative imaging shows tumor rupture with peritoneal fluid. Wilms tumor, favorable histology. COG Stage III.'},
    {'id': 'S03', 'case': '6-year-old male, biopsy prior to chemotherapy violated tumor capsule. Wilms tumor, favorable histology. COG Stage III.'},
    {'id': 'S04', 'case': '4-year-old female, gross spillage during right nephrectomy, no nodes, no mets. Wilms tumor, focal anaplasia. COG Stage III.'},
    {'id': 'S05', 'case': '7-year-old male, contained tumor at surgery (no spillage), positive lymph nodes. Wilms tumor, favorable histology. COG Stage III.'},
]

for c in stretch_cases:
    out = oncology_summary(c['case'])
    c['output'] = out
    # Apply rubric v2 — rule 3 governs these.
    c['human_label'] = 'partially_correct' if 'radiation' not in out.lower() else 'correct'

dataset_v1['cases'].extend([{'id': c['id'], 'case': c['case'], 'output': c['output'], 'human_label': c['human_label']} for c in stretch_cases])
dataset_v1['n_cases'] = len(dataset_v1['cases'])
with open('dataset_v1.json', 'w') as f:
    json.dump(dataset_v1, f, indent=2)
print('Appended 5 stretch cases. New n =', dataset_v1['n_cases'])

## KHCC Connection

At KHCC, the AI Office cannot deploy a clinical assistant on the strength of a demo. A versioned eval dataset with a locked rubric and an inter-annotator kappa is what lets us say *"this assistant performs at X% on cases that look like our patients"* — and what lets us prove the next prompt iteration is actually better, not just different. The dataset you built today is the artifact every future improvement is measured against.